<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/riesgo/notebooks/c4_l5.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C4-L5 · Futuros 2x isolated | 50 trades, margen 100 USDT, lev 2: replica el PnL, cuenta liquidados y verifica el promedio −17,60%.

In [ ]:
# CELDA COLAB-FIRST: correla primero si estas en Google Colab.
# Descarga el CSV del repo; si falla (sin red ), usa el CSV local.
import pandas as pd
from pathlib import Path

ORG = "Emelecto"  # organizacion fija del repo Emelecto/QuantLab
CSV_NOMBRE = "c4_l5.csv"
CSV_URL = f"https://raw.githubusercontent.com/{ORG}/QuantLab/main/web/content/cursos/riesgo/data/{CSV_NOMBRE}"

try:
    df = pd.read_csv(CSV_URL)
    print("CSV descargado desde:", CSV_URL)
except Exception as e:
    print("Uso CSV local (motivo:", str(e)[:80] + ")")
    csv_path = Path("../data") / CSV_NOMBRE
    if not csv_path.exists():
        csv_path = Path(CSV_NOMBRE)  # fallback si corres desde data/
    df = pd.read_csv(csv_path)
print(df.shape)
print(df.head())

In [ ]:
import numpy as np
df["mov"] = (df["precio_salida"] - df["precio_entrada"]) / df["precio_entrada"]
df["pnl_calc"] = (2 * df["mov"] * 100).round(2)
liq = df[df["liquidado"] == 1]
print("Liquidados:", len(liq), "de", len(df))
print("PnL promedio: %.2f%%" % df["pnl_pct"].mean())
print("Mejor: %.2f%% | Peor: %.2f%%" % (df["pnl_pct"].max(), df["pnl_pct"].min()))
print("Perdida maxima por liquidado:", liq["pnl_usdt"].min(), "USDT (el margen, nada mas)")

## Isolated te salva la cuenta | Cada liquidado pierde su margen (100 USDT) y nada más: el fusible se quema, la casa sigue en pie.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df["pnl_pct"], bins=15, color="#f59e0b", edgecolor="#0a0a0b")
ax.axvline(df["pnl_pct"].mean(), color="#f87171", linewidth=2, label="promedio −17,60%")
ax.set_xlabel("PnL por trade (%)")
ax.set_ylabel("Trades")
ax.legend()
plt.show()

In [ ]:
print("Los 6 liquidados perdieron 100 USDT cada uno: el riesgo quedo aislado.")
print("En cross, esa misma racha habria mordido todo el balance.")

In [ ]:
# Chequeos automáticos
assert len(df) == 50, "se esperan 50 trades"
assert df["liquidado"].sum() == 6, "deben ser 6 liquidados"
assert df["pnl_pct"].min() == -100.0, "el peor caso es -100% (liquidado)"
assert df["pnl_pct"].max() == 24.0, "el mejor trade es +24%"
assert abs(df["pnl_pct"].mean() - (-17.60)) < 0.01, "el promedio debe ser -17,60%"
assert (df.loc[df["liquidado"] == 1, "pnl_usdt"] == -100.0).all(), "cada liquidado pierde solo su margen"
print("OK: 2x verificado.")